In [0]:
# osmium은 apt로 설치해야 함, pip으로 안 됨
%sh apt-get install -y osmium-tool

In [0]:
import subprocess
import json
import os

input_pbf = "/dbfs/mnt/bronze/osm/raw/south-korea-latest.osm.pbf"
output_dir = "/dbfs/mnt/silver/seoul_districts"
config_file = "extract_config.json"

os.makedirs(output_dir, exist_ok=True)

# 구별로 osmium을 3번 돌리면 pbf 전체를 3번 읽어야 해서 느림
# multi-extract로 한 번에 처리
extracts_config = {
    "extracts": [
        {
            "output": "gangnam.pbf",
            "bbox": [127.01, 37.45, 127.12, 37.54],
            "description": "Gangnam-gu extraction"
        },
        {
            "output": "seocho.pbf",
            "bbox": [126.97, 37.43, 127.09, 37.52],
            "description": "Seocho-gu extraction"
        },
        {
            "output": "songpa.pbf",
            "bbox": [127.07, 37.46, 127.18, 37.54],
            "description": "Songpa-gu extraction"
        }
    ],
    "directory": output_dir
}

with open(config_file, 'w') as f:
    json.dump(extracts_config, f)

# complete_ways: 경계에 걸친 도로가 잘리지 않게 하려면 필요
cmd = [
    "osmium", "extract",
    "--config", config_file,
    "--strategy", "complete_ways",
    input_pbf,
    "--overwrite"
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print("✅ 추출 완료")
    print(f"저장 경로: {output_dir}")
else:
    print(f"❌ {result.stderr}")

In [0]:
# 마운트 코드는 보안상 뺌
# Secret Scope 또는 환경변수로 따로 관리
raw_path = "/mnt/raw/seoul_api"
# dbutils.fs.ls(raw_path)

In [0]:
# raw 잘 올라왔는지 단건 확인용
df_raw = spark.read.option("multiline", "true").json("/mnt/raw/seoul_api/citydata_20260408_033105.json")
display(df_raw)

In [0]:
# storage_key는 노트북 상단 변수로 주입하거나 Secret Scope 쓸 것
dbutils.fs.mount(
    source = "wasbs://silver@3dtstorage.blob.core.windows.net/",
    mount_point = "/mnt/silver",
    extra_configs = {"fs.azure.account.key.3dtstorage.blob.core.windows.net": storage_key}
)
print("silver 마운트 완료")